# Tutorial 15 — Quantization

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part V — Inference**  
**Follows:** Tutorial 14 (GRPO)  
**Precedes:** Tutorial 16 (Inference: KV Cache, Flash Attention & Speculative Decoding)

---

## What This Tutorial Covers

A trained model lives in float32 — 4 bytes per parameter. The nano model
is ~43MB. A real 7B model is ~28GB. Quantization reduces numerical precision
to shrink the model and speed up inference, with surprisingly little quality
loss if done correctly.

This tutorial covers:

1. **The numerical argument** — what information is in those 32 bits, how
   much of it matters, and why weights tolerate low precision better than
   activations.
2. **Quantization arithmetic** — the affine mapping from float to int,
   scale and zero-point, symmetric vs asymmetric, per-tensor vs per-channel.
3. **Post-Training Quantization (PTQ)** — INT8 weight quantization from
   scratch. No training required.
4. **Calibration** — why you need a small dataset to set scale factors,
   how to collect activation statistics.
5. **INT4 weight quantization** — GPTQ-style block quantization. Why
   4-bit weights work but 4-bit activations do not.
6. **Quantization-Aware Training (QAT)** — fake quantization, straight-
   through estimator, when QAT is worth the cost.
7. **`torch.quantization`** — PyTorch's built-in quantization API,
   where it helps and where it falls short.
8. **Measuring quality degradation** — perplexity on a held-out set
   as a function of bit width. The bit-width vs perplexity tradeoff curve.

---

## 1. The Numerical Argument

A float32 number has 1 sign bit, 8 exponent bits, and 23 mantissa bits.
The 23 mantissa bits give about 7 decimal digits of precision.

For inference, do we need 7 digits of precision per weight? Consider what
a weight actually does: it scales and shifts activations. The *relative*
ordering of weights matters far more than their *absolute* precision. A
weight of 0.3142 and a weight of 0.3141 produce nearly identical outputs
— the difference is 0.03% of the weight's magnitude.

The empirical finding, replicated across many model sizes and architectures:
[**[weights can be quantized to 8 bits with < 1% perplexity increase]{.underline}**.]{.mark}
4-bit weight quantization loses 2–5% perplexity. 3-bit loses noticeably
more. 2-bit degrades rapidly.

Activations are harder: they can take on a much wider range of values
and have *outliers*[^act_quant]

[^act_quant]: Activation outliers were studied in LLM.int8() (Dettmers et al., 2022). A small fraction (~0.1%) of hidden-state dimensions contain values 10–100× larger than typical. These outliers force the quantization scale to be coarse to avoid clipping, wasting the grid resolution on the common case. Mixed-precision approaches keep these outlier dimensions in FP16 and quantize the rest. — a small number of activation values that are far
larger than the rest (a phenomenon studied in LLM.int8()). These outliers
dominate the quantization range and force a coarse grid for the common
values.

The practical consequence: [**weight-only quantization** (INT8 or INT4
weights, FP16/BF16 activations) is the dominant approach for LLM inference.]{.underline}
Full INT8 quantization (both weights and activations) is possible but
requires more careful calibration.

---

## 2. Quantization Arithmetic

### The affine mapping

Quantization maps floating-point values to integers via:

$$x_q = \text{clamp}\!\left(\text{round}\!\left(\frac{x}{s}\right) + z,\; 0,\; 2^b - 1\right)$$

Dequantization recovers an approximation of the original:

$$\hat{x} = s \cdot (x_q - z)$$

where:
- $s > 0$ is the **scale** (the size of one quantization step)
- $z$ is the **zero-point** (the integer that maps to float 0.0)
- $b$ is the bit width (8 for INT8, 4 for INT4)

The quantization error is $x - \hat{x}$ — bounded by $s/2$ (half a step).

### Symmetric vs asymmetric

**Symmetric quantization** forces $z = 0$. The integer range is
$[-2^{b-1}, 2^{b-1}-1]$ (signed) and the mapping is simply $x_q = \text{round}(x/s)$.
Simpler and faster — no zero-point subtraction during inference.

**Asymmetric quantization** allows $z \neq 0$. It uses the full unsigned
range $[0, 2^b - 1]$ and can represent asymmetric distributions more
efficiently. Used when the weight distribution is not centered at zero.

For most LLM weights (which tend to be approximately symmetric around zero),
symmetric quantization is preferred.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from dataclasses import dataclass


def quantize_symmetric(
    x:       torch.Tensor,
    n_bits:  int = 8,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Symmetric per-tensor quantization.

    Returns:
        x_q:   quantized integer tensor (int8 for 8-bit)
        scale: float scale factor
    """
    n_levels = 2 ** (n_bits - 1) - 1   # e.g. 127 for INT8
    abs_max  = x.abs().max().clamp(min=1e-8)
    scale    = abs_max / n_levels

    x_q = torch.clamp(
        torch.round(x / scale),
        -n_levels, n_levels
    ).to(torch.int8 if n_bits <= 8 else torch.int16)

    return x_q, scale


def dequantize_symmetric(
    x_q:   torch.Tensor,
    scale: torch.Tensor,
) -> torch.Tensor:
    """Recover float approximation from quantized tensor."""
    return x_q.float() * scale


def quantization_error(x: torch.Tensor, n_bits: int = 8) -> dict:
    """Measure quantization error statistics."""
    x_q, scale = quantize_symmetric(x, n_bits)
    x_hat      = dequantize_symmetric(x_q, scale)
    err        = (x - x_hat).abs()
    snr        = x.pow(2).mean() / (err.pow(2).mean() + 1e-10)

    return {
        'max_error':    err.max().item(),
        'mean_error':   err.mean().item(),
        'rmse':         err.pow(2).mean().sqrt().item(),
        'snr_db':       10 * torch.log10(snr).item(),
        'scale':        scale.item(),
    }


# Demonstrate: typical weight tensor quantization error
torch.manual_seed(42)
w = torch.randn(768, 768) * 0.02   # typical LLM weight magnitude

for bits in [8, 6, 4, 3, 2]:
    stats = quantization_error(w, bits)
    print(f"INT{bits}: rmse={stats['rmse']:.6f}  "
          f"snr={stats['snr_db']:.1f}dB  "
          f"scale={stats['scale']:.6f}")

### Per-tensor vs per-channel

**Per-tensor quantization** uses a single scale for all weights in a layer.
Simple, but if different output channels have very different weight
magnitudes (common in practice), the large-magnitude channels force a
coarse scale, wasting precision on small-magnitude channels.

[**Per-channel quantization** uses a separate scale per output channel.
Each channel's scale is set by its own max absolute value. More accurate
but requires storing $d_{\text{out}}$ scale factors per layer instead of one.]{.mark}

In [ ]:
def quantize_per_channel(
    weight: torch.Tensor,   # (d_out, d_in)
    n_bits: int = 8,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Per-output-channel symmetric quantization.

    Returns:
        w_q:    (d_out, d_in) quantized weights
        scales: (d_out,) per-channel scale factors
    """
    n_levels = 2 ** (n_bits - 1) - 1
    # Max absolute value per output channel
    abs_max  = weight.abs().max(dim=1).values.clamp(min=1e-8)  # (d_out,)
    scales   = abs_max / n_levels                               # (d_out,)

    # Divide each row by its scale
    w_scaled = weight / scales.unsqueeze(1)   # (d_out, d_in)
    w_q      = torch.clamp(
        torch.round(w_scaled), -n_levels, n_levels
    ).to(torch.int8)

    return w_q, scales


def dequantize_per_channel(
    w_q:    torch.Tensor,   # (d_out, d_in) int8
    scales: torch.Tensor,   # (d_out,)
) -> torch.Tensor:
    return w_q.float() * scales.unsqueeze(1)

---

## 3. Post-Training Quantization (PTQ)

PTQ quantizes a trained model without any additional training. The weights
are fixed — we just change how we store and compute with them.

The simplest PTQ scheme: quantize every `nn.Linear` weight to INT8.
During inference, dequantize to float before the matmul. This gives
2× memory reduction with essentially free quality on 8-bit.

In [ ]:
class QuantizedLinear(nn.Module):
    """
    INT8 weight quantization with float activation.
    Weights stored as int8; dequantized to float at forward time.

    Memory: 1 byte/param (vs 4 bytes for float32) = 4× compression.
    Speed: depends on hardware. On CPU with FBGEMM backend, int8 matmul
           is ~2× faster. On GPU, benefit depends on memory bandwidth.
    """

    def __init__(
        self,
        in_features:  int,
        out_features: int,
        bias:         bool = True,
        n_bits:       int  = 8,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.n_bits       = n_bits

        # Quantized weight stored as int8
        self.register_buffer('weight_q', torch.zeros(out_features, in_features,
                                                       dtype=torch.int8))
        # Per-channel scales
        self.register_buffer('scales', torch.ones(out_features))

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.bias = None

    @classmethod
    def from_linear(
        cls,
        linear:  nn.Linear,
        n_bits:  int = 8,
    ) -> 'QuantizedLinear':
        """Quantize an existing nn.Linear layer."""
        layer = cls(
            linear.in_features, linear.out_features,
            bias=(linear.bias is not None), n_bits=n_bits
        )
        w_q, scales = quantize_per_channel(linear.weight.data, n_bits)
        layer.weight_q.copy_(w_q)
        layer.scales.copy_(scales)
        if linear.bias is not None:
            layer.bias.data.copy_(linear.bias.data)
        return layer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dequantize weights at forward time
        weight_fp = dequantize_per_channel(self.weight_q, self.scales)
        return nn.functional.linear(x, weight_fp.to(x.dtype), self.bias)

    def extra_repr(self) -> str:
        return (f'in={self.in_features}, out={self.out_features}, '
                f'bits={self.n_bits}')


def quantize_model_ptq(
    model:   nn.Module,
    n_bits:  int = 8,
    skip:    list[str] = None,
) -> nn.Module:
    """
    Replace all nn.Linear layers with QuantizedLinear.

    skip: list of name substrings to exclude (e.g. ['lm_head'] to keep
          the output projection in float for better final token quality).
    """
    if skip is None:
        skip = ['lm_head']

    replaced = 0
    for name, module in list(model.named_modules()):
        if any(s in name for s in skip):
            continue
        if not isinstance(module, nn.Linear):
            continue

        parts  = name.split('.')
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr   = parts[-1]

        setattr(parent, attr, QuantizedLinear.from_linear(module, n_bits=n_bits))
        replaced += 1

    n_params  = sum(p.numel() for p in model.parameters())
    n_buffers = sum(b.numel() for b in model.buffers())
    param_mb  = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
    buf_mb    = sum(b.numel() * b.element_size() for b in model.buffers()) / 1e6
    total_mb  = param_mb + buf_mb

    print(f"PTQ INT{n_bits}: replaced {replaced} layers")
    print(f"  Memory: {total_mb:.1f} MB  "
          f"(params: {param_mb:.1f} MB, buffers: {buf_mb:.1f} MB)")
    return model

---

## 4. Calibration

Calibration runs a small set of representative inputs through the model
and collects statistics about the activation ranges. For weight-only
quantization (what we have done so far), calibration is not needed —
weight ranges are fixed after training. For activation quantization,
calibration determines the scale factors for intermediate tensors.

Even for weight-only quantization, calibration is used in more advanced
methods like **GPTQ** (Section 5) and **SmoothQuant**, which adjust
weight quantization based on activation statistics to minimize the
end-to-end quantization error.

In [ ]:
class ActivationCalibrator:
    """
    Hooks into a model and records min/max of all Linear layer inputs
    and outputs over a calibration dataset. Used to set activation
    quantization scales for full INT8 quantization.
    """

    def __init__(self, model: nn.Module):
        self.stats  = {}   # name → {'min': float, 'max': float, 'n': int}
        self.hooks  = []
        self._register_hooks(model)

    def _register_hooks(self, model: nn.Module):
        for name, module in model.named_modules():
            if isinstance(module, (nn.Linear, QuantizedLinear)):
                def make_hook(n):
                    def hook(mod, inp, out):
                        x = inp[0].detach().float()
                        if n not in self.stats:
                            self.stats[n] = {
                                'input_min':  x.min().item(),
                                'input_max':  x.max().item(),
                                'output_min': out.detach().float().min().item(),
                                'output_max': out.detach().float().max().item(),
                                'n_batches':  1,
                            }
                        else:
                            s = self.stats[n]
                            s['input_min']  = min(s['input_min'],  x.min().item())
                            s['input_max']  = max(s['input_max'],  x.max().item())
                            s['output_min'] = min(s['output_min'],
                                                  out.detach().float().min().item())
                            s['output_max'] = max(s['output_max'],
                                                  out.detach().float().max().item())
                            s['n_batches'] += 1
                    return hook
                self.hooks.append(module.register_forward_hook(make_hook(name)))

    def calibrate(self, model, dataloader, device, n_batches=50):
        model.eval()
        with torch.no_grad():
            for i, (x, _) in enumerate(dataloader):
                if i >= n_batches:
                    break
                model(x.to(device))
        print(f"Calibrated on {min(n_batches, i+1)} batches")
        return self.stats

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def print_summary(self):
        print(f"\nActivation ranges (calibrated):")
        print(f"  {'Layer':50s}  {'Input range':20s}  Output range")
        for name, s in list(self.stats.items())[:10]:
            print(f"  {name:50s}  "
                  f"[{s['input_min']:+.3f}, {s['input_max']:+.3f}]  "
                  f"[{s['output_min']:+.3f}, {s['output_max']:+.3f}]")

---

## 5. INT4 Weight Quantization

4-bit quantization halves memory again vs INT8. With 4 bits, each weight
can take only 16 distinct values — intuitively, this seems too coarse.
But in practice, for LLMs with billions of parameters, the effect on
perplexity is modest because:

[1. Individual weights have small effect on output — errors average out]{.mark}
2. Per-channel scaling adapts the quantization grid to each channel's range
3. **Block quantization** further refines by using a separate scale per
   small block of weights within a channel

### Block (group) quantization

Instead of one scale per output channel, use one scale per **group** of
$g$ consecutive input-dimension weights:

```
Channel 0: [w_0, w_1, ..., w_{g-1}] → scale_0
           [w_g, w_{g+1}, ..., w_{2g-1}] → scale_1
           ...
```

Common group sizes: $g = 32, 64, 128$. Smaller groups = more scales
= more accurate but more overhead. $g = 128$ is the standard for
GPTQ-style quantization.

In [ ]:
def quantize_int4_grouped(
    weight:     torch.Tensor,   # (d_out, d_in)
    group_size: int = 128,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    INT4 grouped quantization.
    Each group of `group_size` weights along the input dimension shares
    one scale factor.

    Returns:
        w_q:    (d_out, d_in) quantized to 4-bit range [-8, 7], stored as int8
        scales: (d_out, d_in // group_size) per-group scale factors
    """
    d_out, d_in = weight.shape
    assert d_in % group_size == 0, \
        f"d_in={d_in} must be divisible by group_size={group_size}"

    n_groups = d_in // group_size
    n_levels = 7   # INT4 signed: range [-8, 7]

    # Reshape to (d_out, n_groups, group_size)
    w_grouped = weight.view(d_out, n_groups, group_size)

    # Per-group scale
    abs_max = w_grouped.abs().max(dim=2).values.clamp(min=1e-8)  # (d_out, n_groups)
    scales  = abs_max / n_levels                                   # (d_out, n_groups)

    # Quantize
    w_scaled = w_grouped / scales.unsqueeze(2)   # (d_out, n_groups, group_size)
    w_q      = torch.clamp(torch.round(w_scaled), -8, 7).to(torch.int8)
    w_q      = w_q.view(d_out, d_in)             # (d_out, d_in)

    return w_q, scales


def dequantize_int4_grouped(
    w_q:        torch.Tensor,   # (d_out, d_in) int8
    scales:     torch.Tensor,   # (d_out, n_groups)
    group_size: int = 128,
) -> torch.Tensor:
    d_out, d_in = w_q.shape
    n_groups    = d_in // group_size

    w_grouped = w_q.view(d_out, n_groups, group_size).float()
    w_fp      = w_grouped * scales.unsqueeze(2)   # (d_out, n_groups, group_size)
    return w_fp.view(d_out, d_in)

### Why activations stay in float

The key obstacle to activating activation quantization is **outliers**.
Research on LLM activations (e.g., LLM.int8()) found that a small fraction
(~0.1%) of activation dimensions have values 100× larger than the median.
These outliers force the quantization scale to be large, wasting the entire
4-bit range on rare extreme values while the common values all map to the
same few integers.

[[Weights do not have this problem: weight distributions are approximately
Gaussian with few outliers]{.mark} (weight decay regularizes against large weights).
This is the fundamental reason why **weight-only quantization** works so
much better than full quantization at the same bit width.

---

## 6. Quantization-Aware Training (QAT)

PTQ quantizes after training — the weights were never optimized to be
robust to quantization. QAT inserts quantization into the training loop:
the forward pass simulates quantization, and the backward pass trains
weights that are inherently more quantizable.

### Fake quantization and the [straight-through estimator]{.underline}

During the forward pass, we apply the full quantize-dequantize cycle
(producing $\hat{x} \approx x$ with quantization noise). During the
backward pass, we face a problem: `round()` has zero gradient almost
everywhere. The **straight-through estimator (STE)** simply passes the
gradient through the rounding operation as if it were the identity:

$$\frac{\partial \hat{x}}{\partial x} \approx \mathbf{1}[|x / s| \leq n\_\text{levels}]$$

(1 within the representable range, 0 outside — the clamp has zero gradient
outside and we backpropagate through it honestly).

In [ ]:
class FakeQuantize(torch.autograd.Function):
    """
    Forward: quantize then dequantize (introduces quantization noise).
    Backward: straight-through estimator — pass gradient unchanged
              through the rounding, zero it outside the clamp range.
    """

    @staticmethod
    def forward(ctx, x, scale, n_bits):
        n_levels = 2 ** (n_bits - 1) - 1
        x_scaled = x / scale
        # Save mask for backward (zero gradient outside clamp range)
        ctx.save_for_backward(
            (x_scaled.abs() <= n_levels).float()
        )
        x_q   = torch.clamp(torch.round(x_scaled), -n_levels, n_levels)
        return x_q * scale   # dequantized

    @staticmethod
    def backward(ctx, grad_output):
        mask, = ctx.saved_tensors
        # STE: pass gradient through rounding; zero outside clamp
        return grad_output * mask, None, None


def fake_quantize(x: torch.Tensor, scale: torch.Tensor, n_bits: int) -> torch.Tensor:
    return FakeQuantize.apply(x, scale, n_bits)


class QATLinear(nn.Module):
    """
    Linear layer with fake quantization in the forward pass.
    Weights are trained in float but experience quantization noise
    during each forward pass — making them robust to PTQ at inference.
    """

    def __init__(
        self,
        in_features:  int,
        out_features: int,
        bias:         bool = True,
        n_bits:       int  = 8,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.n_bits       = n_bits
        self.weight       = nn.Parameter(torch.empty(out_features, in_features))
        self.bias         = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.kaiming_uniform_(self.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            # Compute per-channel scale from current weights
            n_levels = 2 ** (self.n_bits - 1) - 1
            scales   = self.weight.abs().max(dim=1).values.clamp(min=1e-8) / n_levels
            # Apply fake quantization row-by-row
            w_fq = torch.stack([
                fake_quantize(self.weight[i], scales[i], self.n_bits)
                for i in range(self.out_features)
            ])
            return nn.functional.linear(x, w_fq, self.bias)
        else:
            # At eval time, use real quantized weights (PTQ)
            w_q, scales = quantize_per_channel(self.weight.data, self.n_bits)
            w_fp        = dequantize_per_channel(w_q, scales).to(x.dtype)
            return nn.functional.linear(x, w_fp, self.bias)

### When is QAT worth the cost?

QAT requires re-running fine-tuning with quantization noise — typically
10–20% of the original training compute. The quality improvement over PTQ
is significant at 4-bit but modest at 8-bit:

| Method | INT8 perplexity delta | INT4 perplexity delta |
|---|---|---|
| PTQ (no calibration) | +0.5–2% | +5–15% |
| PTQ (calibrated) | +0.1–0.5% | +2–8% |
| QAT | +0.05–0.2% | +0.5–2% |

[[For INT8: PTQ with calibration is almost always good enough. Skip QAT.]{.mark}
For INT4: QAT is worth considering if you need the best possible quality,
especially for a production model that will serve many requests.

---

## 7. Measuring Quality Degradation

Perplexity is the standard metric for quantization quality assessment.
Lower is better. Measure it on a held-out calibration corpus:

In [ ]:
@torch.no_grad()
def evaluate_perplexity(
    model,
    dataloader,
    device:    torch.device,
    n_batches: int = 100,
    dtype:     torch.dtype = torch.float32,
) -> float:
    """
    Compute perplexity on a dataloader.
    Perplexity = exp(cross-entropy loss).
    """
    model.eval()
    total_loss   = 0.0
    total_tokens = 0

    for i, (x, y) in enumerate(dataloader):
        if i >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=dtype):
            _, loss = model(x, y)
        n_tokens      = (y != -100).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss    = total_loss / max(total_tokens, 1)
    perplexity  = np.exp(avg_loss)
    return perplexity


def quantization_quality_sweep(
    model_path: str,
    dataloader,
    device:     torch.device,
):
    """
    Measure perplexity at each bit width and print a comparison table.
    """
    from tutorial_02 import GPT, NanoGPTConfig
    config = NanoGPTConfig()

    results = {}

    # Baseline: full float32
    model_fp32 = GPT(config).to(device)
    ckpt = torch.load(model_path, map_location=device)
    model_fp32.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    ppl_fp32 = evaluate_perplexity(model_fp32, dataloader, device)
    results['FP32'] = ppl_fp32
    print(f"FP32     : perplexity = {ppl_fp32:.3f}")

    # INT8 PTQ
    model_int8 = GPT(config).to(device)
    model_int8.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    quantize_model_ptq(model_int8, n_bits=8)
    ppl_int8 = evaluate_perplexity(model_int8, dataloader, device)
    results['INT8'] = ppl_int8
    delta = 100 * (ppl_int8 - ppl_fp32) / ppl_fp32
    print(f"INT8 PTQ : perplexity = {ppl_int8:.3f}  ({delta:+.2f}%)")

    # INT4 PTQ (per-channel, group_size=32 for nano model)
    # For nano, d_in=384 which is divisible by 32
    model_int4 = GPT(config).to(device)
    model_int4.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)

    replaced = 0
    for name, module in list(model_int4.named_modules()):
        if not isinstance(module, nn.Linear) or 'lm_head' in name:
            continue
        parts  = name.split('.')
        parent = model_int4
        for part in parts[:-1]:
            parent = getattr(parent, part)

        w = module.weight.data
        # Pad d_in to be divisible by 32 if needed
        d_out, d_in = w.shape
        gs = 32 if d_in % 32 == 0 else d_in   # fallback: whole channel
        w_q, scales = quantize_int4_grouped(w, group_size=gs)

        # Wrap as a callable that dequantizes at forward time
        class Int4Linear(nn.Module):
            def __init__(self, w_q, scales, gs, bias):
                super().__init__()
                self.register_buffer('w_q', w_q)
                self.register_buffer('scales', scales)
                self.gs = gs
                self.bias_param = bias
            def forward(self, x):
                w = dequantize_int4_grouped(self.w_q, self.scales, self.gs)
                return nn.functional.linear(x, w.to(x.dtype), self.bias_param)

        setattr(parts[-2] if len(parts) > 1 else model_int4,
                parts[-1],
                Int4Linear(w_q, scales, gs, module.bias))
        replaced += 1

    ppl_int4 = evaluate_perplexity(model_int4, dataloader, device)
    results['INT4'] = ppl_int4
    delta = 100 * (ppl_int4 - ppl_fp32) / ppl_fp32
    print(f"INT4 PTQ : perplexity = {ppl_int4:.3f}  ({delta:+.2f}%)")

    print(f"\n{'Bit width':12s}  {'Perplexity':12s}  {'Delta':10s}  "
          f"{'Model size (est.)':20s}")
    n_params = sum(p.numel() for p in model_fp32.parameters())
    for label, ppl in results.items():
        delta  = 100 * (ppl - ppl_fp32) / ppl_fp32
        bits   = {'FP32': 32, 'INT8': 8, 'INT4': 4}[label]
        size_mb = n_params * bits / 8 / 1e6
        print(f"  {label:10s}  {ppl:12.3f}  {delta:+9.2f}%  {size_mb:8.1f} MB")

    return results

---

## 8. Model Size Verification

After quantization, verify the actual memory reduction:

In [ ]:
def model_size_mb(model: nn.Module) -> float:
    """Total size of all parameters and buffers in MB."""
    param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / 1e6


def quantization_summary(model: nn.Module):
    """Print a per-layer quantization summary."""
    print(f"\nQuantization Summary")
    print(f"{'─'*60}")
    total_params  = 0
    quant_params  = 0

    for name, module in model.named_modules():
        if isinstance(module, (nn.Linear, QuantizedLinear)):
            n = (module.weight_q if hasattr(module, 'weight_q')
                 else module.weight).numel()
            is_q = isinstance(module, QuantizedLinear)
            bits = module.n_bits if hasattr(module, 'n_bits') else 32
            total_params += n
            if is_q:
                quant_params += n
            print(f"  {'Q' if is_q else 'F'} {name:45s}  "
                  f"{n/1e3:6.1f}K  INT{bits if is_q else 32}")

    print(f"{'─'*60}")
    print(f"  Total params:      {total_params/1e6:.2f}M")
    print(f"  Quantized params:  {quant_params/1e6:.2f}M  "
          f"({100*quant_params/total_params:.1f}%)")
    print(f"  Model size:        {model_size_mb(model):.1f} MB")

---

## Summary

| Concept | Key detail |
|---|---|
| Why weights tolerate low precision | Relative ordering matters more than absolute value. Errors average across many weights. |
| Affine mapping | $x_q = \text{round}(x/s) + z$, $\hat{x} = s(x_q - z)$. Error bounded by $s/2$. |
| Symmetric quantization | $z=0$, signed range, simpler. Preferred for LLM weights (near-symmetric distribution). |
| Per-channel scaling | One scale per output channel. Adapts to channel-wise magnitude variation. |
| Per-group (block) scaling | One scale per $g$ weights within a channel. Standard for INT4 ($g=128$). |
| PTQ | Quantize after training. No training needed. INT8: near-lossless. |
| Calibration | Needed for activation quantization. Collect min/max over representative inputs. |
| INT4 grouped | 4-bit range $[-8,7]$, group size 128. 4× memory vs FP32, 2–8% perplexity increase. |
| Why activations stay float | Outlier activations (100× median) force coarse quantization grid for common values. |
| Straight-through estimator | $\partial\hat{x}/\partial x \approx 1$ through round. Zero outside clamp range. |
| QAT vs PTQ | QAT worth it for INT4. PTQ + calibration is sufficient for INT8. |
| Skip lm_head | Output projection quantization disproportionately hurts perplexity. Keep in float. |

---

## Exercises

**1.** Run `quantization_error` for a typical LLM weight tensor
(`torch.randn(768, 768) * 0.02`) at bits ∈ {8, 6, 4, 3, 2}.
Plot RMSE vs bit width on a log scale. Identify the "knee" — the bit
width where error starts growing rapidly. Is it different for a weight
tensor with outliers (`w[0, 0] = 10.0`)?

**2.** Implement `compare_per_tensor_vs_per_channel`: quantize the same
weight matrix with per-tensor and per-channel INT8, compute the RMSE
of each, and plot the per-channel weight magnitude distribution. Confirm
that per-channel is always at least as good as per-tensor, and that
the gap is larger when the channel magnitude distribution has high variance.

**3.** Run `quantization_quality_sweep` on the trained nano model. Record
the perplexity table. Then run best-of-N generation (Tutorial 13) with
the FP32 and INT8 models and compare the reward scores of their outputs.
Confirm that the reward difference between FP32 and INT8 is small
relative to the difference between the pretrained and SFT models.

**4.** Implement and verify the straight-through estimator: create a simple
one-layer model with `QATLinear`, train it on a toy regression task for
100 steps, then apply PTQ and compare predictions. The QAT-trained model
should degrade less under PTQ than a model trained with standard
`nn.Linear`.

**5.** Implement **dynamic quantization**: instead of setting the scale from
the weight distribution at PTQ time, set it from the *actual input*
distribution at runtime. This means the scale changes every forward pass.
Dynamic quantization is slower than static PTQ but more accurate for
layers with highly variable input distributions (e.g., the first layer).
Compare perplexity of static vs dynamic INT8 quantization.

**6.** Measure the actual memory and throughput impact of INT8 quantization
on your hardware. Load the full FP32 model and the INT8 model, run 100
identical forward passes, and report: (a) peak GPU memory (via
`torch.cuda.max_memory_allocated()`), (b) mean tokens/sec throughput.
On a memory-bandwidth-limited GPU (most consumer cards), INT8 should
show a meaningful throughput improvement. On a compute-limited GPU
(A100), the benefit may be smaller.